In [4]:
%pip install holidays

Note: you may need to restart the kernel to use updated packages.


In [5]:
%cd Proyecto_Big_Data

[Errno 2] No such file or directory: 'Proyecto_Big_Data'
/home/jovyan/work/Proyecto_Big_Data


In [6]:
import boto3
import os

# --- CONFIGURACIÓN AWS ---
S3_BUCKET = "proyecto-demanda-clima" # Tu nombre de bucket sin espacios
s3_client = boto3.client("s3")

def subir_a_s3(ruta_archivo_local, carpeta_destino_s3):
    """
    Sube un archivo a S3 y confirma la operación.
    carpeta_destino_s3 debe ser algo como 'bronce/clima/'
    """
    nombre_archivo = os.path.basename(ruta_archivo_local)
    key_s3 = f"{carpeta_destino_s3}{nombre_archivo}"
    
    try:
        s3_client.upload_file(ruta_archivo_local, S3_BUCKET, key_s3)
        print(f"✅ Archivo {nombre_archivo} subido a s3://{S3_BUCKET}/{key_s3}")
    except Exception as e:
        print(f"❌ Error al subir {nombre_archivo}: {e}")

In [7]:
import os
import subprocess
import pandas as pd
import requests
from pathlib import Path
from datetime import datetime
import re

# --- CONFIGURACIÓN DE RUTAS ---
UNRAR_EXE = '/usr/bin/unrar' 
FOLDER_BRONCE = "./datos_clima_bronce"
FOLDER_EXTRACT = "./datos_clima_temp"
BASE_URL = "https://datosclima.es/capturadatos/"

Path(FOLDER_BRONCE).mkdir(parents=True, exist_ok=True)
Path(FOLDER_EXTRACT).mkdir(parents=True, exist_ok=True)

# --- FUNCIONES DE LIMPIEZA Y APOYO ---

def limpiar_valor(v):
    """
    Limpia strings tipo '19.5 (14:40)' y los convierte en floats (19.5).
    Si no hay número, devuelve None.
    """
    if pd.isna(v) or v == '' or v == 'nan': 
        return None
    # Buscamos el primer número decimal o entero en el texto
    match = re.search(r"[-+]?\d*\.\d+|\d+", str(v))
    return float(match.group()) if match else None

def extraer_fecha_del_nombre_archivo(nombre_f, anio_contexto, mes_contexto):
    """
    Extrae el día del nombre del archivo (ej: '05.xls') y monta la fecha completa.
    """
    numeros = re.findall(r'\d+', nombre_f)
    dia = "01"
    if numeros:
        # El primer número suele ser el día en estos archivos
        posible_dia = numeros[0].zfill(2)
        if int(posible_dia) <= 31:
            dia = posible_dia
            
    return f"{anio_contexto}-{str(mes_contexto).zfill(2)}-{dia}"

# --- PROCESO PRINCIPAL ---

def ingesta_incremental_final(anio_inicio=2013):
    ahora = datetime.now()
    
    for anio in range(anio_inicio, ahora.year + 1):
        nombre_csv_anual = f"clima_{anio}_bronce.csv"
        
        if os.path.exists(nombre_csv_anual) and anio < ahora.year:
            print(f"⏩ Año {anio} ya existe. Saltando...")
            continue
            
        print(f"\n--- 🔄 PROCESANDO AÑO: {anio} ---")
        lista_anual = []
        mes_inicio = 5 if anio == 2013 else 1
        
        for mes in range(mes_inicio, 13):
            if anio == ahora.year and mes > ahora.month:
                break
                
            nombre_rar = f"Aemet{anio}-{mes:02d}.rar"
            url = f"{BASE_URL}{nombre_rar}"
            path_rar = os.path.join(FOLDER_BRONCE, nombre_rar)
            
            try:
                res = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
                if res.status_code == 200:
                    with open(path_rar, "wb") as f:
                        f.write(res.content)
                    
                    subprocess.run([UNRAR_EXE, 'e', '-o+', path_rar, FOLDER_EXTRACT], capture_output=True)
                    
                    # Ordenamos los ficheros alfabéticamente para que el día 1 vaya antes que el 2
                    ficheros = sorted([f for f in os.listdir(FOLDER_EXTRACT) if f.endswith(('.xls', '.xlsx'))])
                    
                    # --- MEJORA: Asignación por orden de archivo ---
                    for i, f in enumerate(ficheros):
                        path_f = os.path.join(FOLDER_EXTRACT, f)
                        dia = str(i + 1).zfill(2) # El primer archivo es el día 01, el segundo 02...
                        fecha_registro = f"{anio}-{str(mes).zfill(2)}-{dia}"
                        
                        try:
                            df_preview = pd.read_excel(path_f, header=None, nrows=10).astype(str)
                            idx_header = 4
                            for fila_idx, row in df_preview.iterrows():
                                if "estación" in " ".join(row.values).lower():
                                    idx_header = fila_idx
                                    break
                            
                            df_dia = pd.read_excel(path_f, skiprows=idx_header)
                            df_dia.columns = [str(c).replace('\n', ' ').strip() for c in df_dia.columns]
                            df_dia = df_dia[df_dia['Estación'].notna()].copy()
                            
                            # Asignamos la fecha calculada por posición
                            df_dia['fecha'] = fecha_registro
                            
                            cols_clima = ['Temperatura máxima (ºC)', 'Temperatura mínima (ºC)', 'Temperatura media (ºC)', 
                                         'Precipitación (mm)', 'Racha (km/h)', 'Velocidad máxima (km/h)']
                            for col in cols_clima:
                                if col in df_dia.columns:
                                    df_dia[col] = df_dia[col].apply(limpiar_valor)
                            
                            lista_anual.append(df_dia)
                        except Exception as e:
                            print(f"⚠️ Error en día {dia}: {e}")
                        
                        if os.path.exists(path_f): os.remove(path_f)
                    
                    if os.path.exists(path_rar): os.remove(path_rar)
                    print(f"✅ Mes {mes:02d}/{anio} completado ({len(ficheros)} días).")
            except Exception as e:
                print(f"💥 Error: {e}")

        if lista_anual:
            df_final = pd.concat(lista_anual, ignore_index=True)
            # Limpieza de columnas fantasma
            df_final = df_final.loc[:, ~df_final.columns.str.contains('^Unnamed')]
            # Reordenar columnas
            cols = ['fecha'] + [c for c in df_final.columns if c != 'fecha']
            
            # 1. Guardamos el CSV localmente (como hacías antes)
            df_final[cols].to_csv(nombre_csv_anual, index=False)
            print(f"💾 Archivo local creado: {nombre_csv_anual}")
            
            # 2. 🚀 SUBIDA A S3 (Lo nuevo)
            # Llamamos a la función que definimos antes
            subir_a_s3(nombre_csv_anual, "bronce/clima/")

# Ejecutar el motor
ingesta_incremental_final()


--- 🔄 PROCESANDO AÑO: 2013 ---
✅ Mes 05/2013 completado (55 días).
✅ Mes 06/2013 completado (30 días).
✅ Mes 07/2013 completado (31 días).
✅ Mes 08/2013 completado (31 días).
✅ Mes 09/2013 completado (30 días).
✅ Mes 10/2013 completado (31 días).
✅ Mes 11/2013 completado (29 días).
✅ Mes 12/2013 completado (31 días).
💾 Archivo local creado: clima_2013_bronce.csv
❌ Error al subir clima_2013_bronce.csv: Failed to upload clima_2013_bronce.csv to proyecto-demanda-clima/bronce/clima/clima_2013_bronce.csv: An error occurred (ExpiredToken) when calling the CreateMultipartUpload operation: The provided token has expired.

--- 🔄 PROCESANDO AÑO: 2014 ---
✅ Mes 01/2014 completado (31 días).
✅ Mes 02/2014 completado (28 días).
✅ Mes 03/2014 completado (31 días).
⚠️ Error en día 07: Excel file format cannot be determined, you must specify an engine manually.
✅ Mes 04/2014 completado (29 días).
✅ Mes 05/2014 completado (31 días).
✅ Mes 06/2014 completado (30 días).
✅ Mes 07/2014 completado (31 días

/tmp/ipykernel_5875/181131120.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(lista_anual, ignore_index=True)


💾 Archivo local creado: clima_2015_bronce.csv
❌ Error al subir clima_2015_bronce.csv: Failed to upload clima_2015_bronce.csv to proyecto-demanda-clima/bronce/clima/clima_2015_bronce.csv: An error occurred (ExpiredToken) when calling the CreateMultipartUpload operation: The provided token has expired.

--- 🔄 PROCESANDO AÑO: 2016 ---
✅ Mes 01/2016 completado (31 días).
✅ Mes 02/2016 completado (29 días).
✅ Mes 03/2016 completado (31 días).
✅ Mes 04/2016 completado (30 días).
✅ Mes 05/2016 completado (31 días).
✅ Mes 06/2016 completado (30 días).
✅ Mes 07/2016 completado (30 días).
✅ Mes 08/2016 completado (31 días).
✅ Mes 09/2016 completado (30 días).
✅ Mes 10/2016 completado (31 días).
✅ Mes 11/2016 completado (30 días).
✅ Mes 12/2016 completado (31 días).
💾 Archivo local creado: clima_2016_bronce.csv
❌ Error al subir clima_2016_bronce.csv: Failed to upload clima_2016_bronce.csv to proyecto-demanda-clima/bronce/clima/clima_2016_bronce.csv: An error occurred (ExpiredToken) when calling t

/tmp/ipykernel_5875/181131120.py:119: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(lista_anual, ignore_index=True)


💾 Archivo local creado: clima_2019_bronce.csv
❌ Error al subir clima_2019_bronce.csv: Failed to upload clima_2019_bronce.csv to proyecto-demanda-clima/bronce/clima/clima_2019_bronce.csv: An error occurred (ExpiredToken) when calling the CreateMultipartUpload operation: The provided token has expired.

--- 🔄 PROCESANDO AÑO: 2020 ---
✅ Mes 01/2020 completado (31 días).
✅ Mes 02/2020 completado (29 días).
✅ Mes 03/2020 completado (28 días).


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import glob

# 1. Buscamos los archivos anuales que generó tu primer script
ARCHIVOS = sorted(glob.glob("clima_*_bronce.csv"))
LISTA_FINAL = []

print("🚀 Iniciando unificación respetando fechas originales...")

for f in ARCHIVOS:
    try:
        # Leemos el CSV. Como ya está limpio de la fase anterior, no saltamos filas
        df = pd.read_csv(f)
        
        # Verificamos si la columna fecha existe
        if 'fecha' in df.columns:
            # Convertimos a datetime por seguridad
            df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
            
            # ELIMINAMOS filas donde la fecha sea NaT (por si hay basura al final del archivo)
            df = df.dropna(subset=['fecha'])
            
            LISTA_FINAL.append(df)
            print(f"✅ {f} unido: desde {df['fecha'].min().date()} hasta {df['fecha'].max().date()}")
        else:
            print(f"⚠️ {f} no tiene columna 'fecha'. Revisa la fase de ingesta.")

    except Exception as e:
        print(f"❌ Error en {f}: {e}")

# 3. Concatenación Final
if LISTA_FINAL:
    df_maestro = pd.concat(LISTA_FINAL, ignore_index=True)
    
    # Ordenar por fecha y estación para que el dataset sea profesional
    df_maestro = df_maestro.sort_values(by=['fecha', 'Estación']).reset_index(drop=True)
    
    df_maestro.to_csv("clima_plata_unificado.csv", index=False)
    print(f"\n--- ✨ UNIFICACIÓN TOTAL COMPLETADA ---")
    print(f"Total de registros: {len(df_maestro)}")
    print(f"Rango: {df_maestro['fecha'].min()} >>> {df_maestro['fecha'].max()}")

🚀 Iniciando unificación respetando fechas originales...
✅ clima_2013_bronce.csv unido: desde 2013-05-01 hasta 2013-12-31
✅ clima_2014_bronce.csv unido: desde 2014-01-01 hasta 2014-12-31
✅ clima_2015_bronce.csv unido: desde 2015-01-01 hasta 2015-12-31
✅ clima_2016_bronce.csv unido: desde 2016-01-01 hasta 2016-12-31
✅ clima_2017_bronce.csv unido: desde 2017-01-01 hasta 2017-12-31
✅ clima_2018_bronce.csv unido: desde 2018-01-01 hasta 2018-12-31
✅ clima_2019_bronce.csv unido: desde 2019-01-01 hasta 2019-12-31
✅ clima_2020_bronce.csv unido: desde 2020-01-01 hasta 2020-12-31
✅ clima_2021_bronce.csv unido: desde 2021-01-01 hasta 2021-12-31
✅ clima_2022_bronce.csv unido: desde 2022-01-01 hasta 2022-12-31
✅ clima_2023_bronce.csv unido: desde 2023-01-01 hasta 2023-12-31
✅ clima_2024_bronce.csv unido: desde 2024-01-01 hasta 2024-12-31
✅ clima_2025_bronce.csv unido: desde 2025-01-01 hasta 2025-12-31
✅ clima_2026_bronce.csv unido: desde 2026-01-01 hasta 2026-03-31

--- ✨ UNIFICACIÓN TOTAL COMPLETAD

In [ ]:
df_preview = pd.read_csv("clima_plata_unificado.csv")


In [ ]:
display(df_preview.head(250))

,fecha,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm)
0,2013-05-01,A Cañiza,Pontevedra,15.2,10.3,12.8,NaN,NaN,19.0,10.4,3.8,0.8,4.0
1,2013-05-01,A Coruña,A Coruña,19.7,15.0,17.4,66.0,31.0,1.2,1.0,0.0,0.2,0.0
2,2013-05-01,A Coruña Aeropuerto,A Coruña,19.2,14.8,17.0,68.0,39.0,0.1,0.0,0.0,0.1,0.0
3,2013-05-01,A Estrada,Pontevedra,18.2,12.5,15.4,NaN,NaN,24.2,16.2,0.6,0.6,6.8
4,2013-05-01,A Gudiña,Ourense,15.2,8.4,11.8,NaN,NaN,5.4,0.8,2.0,0.0,2.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,2013-05-01,Elgeta,Gipuzkoa,20.9,13.7,17.3,24.0,9.0,NaN,NaN,0.0,0.0,0.0
246,2013-05-01,Elgoibar,Gipuzkoa,22.6,14.1,18.4,36.0,14.0,0.1,0.0,0.1,0.0,0.0
247,2013-05-01,Enciso,La Rioja,20.0,11.5,15.8,36.0,22.0,0.0,0.0,0.0,0.0,0.0
248,2013-05-01,Enguera,València/Valencia,26.1,14.3,20.2,NaN,NaN,0.0,0.0,0.0,0.0,0.0


In [ ]:
import holidays

# Definimos el rango de tu proyecto
fecha_inicio = '2013-05-01'
fecha_fin = '2026-12-31'

# 1. Creamos el rango de fechas
fechas = pd.date_range(start=fecha_inicio, end=fecha_fin)
df_festivos = pd.DataFrame({'fecha': fechas})

# 2. Cargamos los festivos oficiales de España (incluye las CCAA)
# Seleccionamos 'ESP' para España y podemos añadir la provincia (ej: 'CL' para Castilla y León)
festivos_esp = holidays.Spain(years=range(2013, 2027), subdiv='CL')

# 3. Aplicamos la lógica al DataFrame
df_festivos['nombre_festivo'] = df_festivos['fecha'].map(festivos_esp)
df_festivos['es_festivo'] = df_festivos['nombre_festivo'].notna().astype(int)

# 4. Añadimos el tipo de día (Laboral / Fin de semana)
df_festivos['dia_semana'] = df_festivos['fecha'].dt.dayofweek
df_festivos['tipo_dia'] = 'Laboral'
df_festivos.loc[df_festivos['dia_semana'] >= 5, 'tipo_dia'] = 'Fin de Semana'
df_festivos.loc[df_festivos['es_festivo'] == 1, 'tipo_dia'] = 'Festivo'

# Guardamos el archivo "oro" de calendario
df_festivos.to_csv("festivos_espana_real.csv", index=False)

print("✅ Calendario Real generado con la librería 'holidays'")
print(df_festivos[df_festivos['es_festivo'] == 1])
display(df_festivos.head(10))

nombre_festivos = "festivos_espana_bronce.csv"
df_festivos.to_csv(nombre_festivos, index=False)

# 🚀 SUBIDA A LA CAPA BRONCE EN S3
subir_a_s3(nombre_festivos, "bronce/festivos/")

✅ Calendario Real generado con la librería 'holidays'
          fecha                     nombre_festivo  es_festivo  dia_semana  \
0    2013-05-01                          Labor Day           1           2   
106  2013-08-15                     Assumption Day           1           3   
164  2013-10-12                       National Day           1           5   
184  2013-11-01                    All Saints' Day           1           4   
219  2013-12-06                   Constitution Day           1           4   
...         ...                                ...         ...         ...   
4912 2026-10-12                       National Day           1           0   
4933 2026-11-02   Monday following All Saints' Day           1           0   
4968 2026-12-07  Monday following Constitution Day           1           0   
4969 2026-12-08              Immaculate Conception           1           1   
4986 2026-12-25                      Christmas Day           1           4   

     tipo

,fecha,nombre_festivo,es_festivo,dia_semana,tipo_dia
0,2013-05-01,Labor Day,1,2,Festivo
1,2013-05-02,NaN,0,3,Laboral
2,2013-05-03,NaN,0,4,Laboral
3,2013-05-04,NaN,0,5,Fin de Semana
4,2013-05-05,NaN,0,6,Fin de Semana
5,2013-05-06,NaN,0,0,Laboral
6,2013-05-07,NaN,0,1,Laboral
7,2013-05-08,NaN,0,2,Laboral
8,2013-05-09,NaN,0,3,Laboral
9,2013-05-10,NaN,0,4,Laboral


In [ ]:

# 1. Cargamos los dos datasets
df_clima = pd.read_csv("clima_plata_unificado.csv")
df_festivos = pd.read_csv("festivos_espana_real.csv")

# 2. IMPORTANTE: Asegurar que la columna 'fecha' es del mismo tipo en ambos
# Si no lo hacemos, el merge puede fallar o dar resultados vacíos
df_clima['fecha'] = pd.to_datetime(df_clima['fecha'])
df_festivos['fecha'] = pd.to_datetime(df_festivos['fecha'])

# 3. Realizamos la unión (Merge)
# Usamos 'left' para mantener todas las filas de clima y añadir los festivos donde coincidan
df_unificado = pd.merge(df_clima, df_festivos, on='fecha', how='left')

# 4. Verificación de seguridad
# Si algún festivo sale como NaN (porque no estaba en el rango), lo rellenamos como día Laboral
df_unificado['es_festivo'] = df_unificado['es_festivo'].fillna(0).astype(int)
df_unificado['tipo_dia'] = df_unificado['tipo_dia'].fillna('Laboral')

# 5. Guardamos el resultado (Este ya es un dataset de nivel "Plata Avanzada")
df_unificado.to_csv("clima_festivos_consolidado.csv", index=False)

print("✅ Unión completada con éxito.")
print(f"Total de registros finales: {len(df_unificado)}")


✅ Unión completada con éxito.
Total de registros finales: 3753816


In [ ]:
display(df_unificado.head(1000000))

,fecha,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),nombre_festivo,es_festivo,dia_semana,tipo_dia
0,2013-05-01,A Cañiza,Pontevedra,15.2,10.3,12.8,NaN,NaN,19.0,10.4,3.8,0.8,4.0,Labor Day,1,2,Festivo
1,2013-05-01,A Coruña,A Coruña,19.7,15.0,17.4,66.0,31.0,1.2,1.0,0.0,0.2,0.0,Labor Day,1,2,Festivo
2,2013-05-01,A Coruña Aeropuerto,A Coruña,19.2,14.8,17.0,68.0,39.0,0.1,0.0,0.0,0.1,0.0,Labor Day,1,2,Festivo
3,2013-05-01,A Estrada,Pontevedra,18.2,12.5,15.4,NaN,NaN,24.2,16.2,0.6,0.6,6.8,Labor Day,1,2,Festivo
4,2013-05-01,A Gudiña,Ourense,15.2,8.4,11.8,NaN,NaN,5.4,0.8,2.0,0.0,2.6,Labor Day,1,2,Festivo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999995,2016-11-25,Beariz,Ourense,9.4,0.5,4.9,NaN,NaN,0.4,0.2,0.0,0.0,0.2,NaN,0,4,Laboral
999996,2016-11-25,Beasain,Gipuzkoa,12.2,-0.3,5.9,34.0,19.0,0.0,0.0,0.0,0.0,0.0,NaN,0,4,Laboral
999997,2016-11-25,Becerreá,Lugo,3.7,-0.8,1.5,44.0,19.0,15.0,0.0,1.0,13.2,0.8,NaN,0,4,Laboral
999998,2016-11-25,Bello,Teruel,6.4,-4.3,1.0,32.0,24.0,0.4,0.0,0.2,0.0,0.2,NaN,0,4,Laboral
